# Task 1: Air Quality Forecasting
**Candidate:** Subhankar

Analyse historical air-quality measurements (UCI Air Quality Dataset) and build a model to predict a future air-quality measurement.

## Setup — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import urllib.request
import zipfile

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)

os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)

print('Setup complete.')

## Part A — Data Understanding & Preprocessing

**About the dataset**: UCI Air Quality dataset contains ~9,358 hourly averaged readings from gas sensors deployed in an Italian city, along with ground-truth pollutant concentrations (CO, NOx, NO2, Benzene, etc.) and weather data (temperature, humidity). Missing/invalid readings in the raw dataset are marked as `-200`.

In [ ]:
# Download the dataset (UCI Machine Learning Repository)
url = 'https://archive.ics.uci.edu/static/public/360/air+quality.zip'
zip_path = 'AirQualityUCI.zip'

if not os.path.exists('AirQualityUCI.csv'):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')
    print('Downloaded and extracted dataset.')
else:
    print('Dataset already present.')

In [ ]:
# Load dataset (semicolon-separated, comma as decimal separator)
df = pd.read_csv('AirQualityUCI.csv', sep=';', decimal=',')

# Drop fully empty trailing columns/rows commonly present in this file
df = df.dropna(axis=1, how='all')
df = df.dropna(axis=0, how='all')

print('Shape:', df.shape)
df.head()

In [ ]:
print('Columns:', list(df.columns))
print('\nData types:\n', df.dtypes)

In [ ]:
# Combine Date and Time into a proper datetime index
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H.%M.%S')
df = df.set_index('Datetime').sort_index()
df = df.drop(columns=['Date', 'Time'])

print('Time range:', df.index.min(), 'to', df.index.max())
df.head()

In [ ]:
# -200 marks missing/invalid readings in this dataset — replace with NaN
df = df.replace(-200, np.nan)
df = df.replace(-200.0, np.nan)

missing_pct = df.isna().mean().sort_values(ascending=False) * 100
print('Missing value percentage per column:')
print(missing_pct)

In [ ]:
# Drop columns that are almost entirely missing (e.g. NMHC(GT) is famously ~90% missing)
cols_to_drop = missing_pct[missing_pct > 50].index.tolist()
print('Dropping near-empty columns:', cols_to_drop)
df = df.drop(columns=cols_to_drop)

# Check duplicate timestamps
print('Duplicate index entries:', df.index.duplicated().sum())
df = df[~df.index.duplicated(keep='first')]

print('Shape after cleanup:', df.shape)

In [ ]:
# Handle remaining missing values — time-based interpolation preserves temporal trend
# better than mean/median fill or dropping rows, since order matters for a time series.
df = df.interpolate(method='time', limit_direction='both')

print('Remaining missing values after interpolation:', df.isna().sum().sum())
print('Final clean dataset shape:', df.shape)

## Part B — Exploratory Data Analysis

In [ ]:
# Distribution of key pollutants
key_cols = [c for c in ['CO(GT)', 'NO2(GT)', 'C6H6(GT)', 'NOx(GT)'] if c in df.columns]

fig, axes = plt.subplots(1, len(key_cols), figsize=(5 * len(key_cols), 4))
if len(key_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, key_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f'Distribution of {col}')
plt.tight_layout()
plt.savefig('results/task1_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Time series over the full period
target_col = 'CO(GT)' if 'CO(GT)' in df.columns else key_cols[0]

plt.figure(figsize=(14, 4))
df[target_col].plot()
plt.title(f'{target_col} over Time')
plt.xlabel('Datetime')
plt.ylabel(target_col)
plt.tight_layout()
plt.savefig('results/task1_timeseries.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Average pollutant level by hour of day — reveals daily (e.g. rush hour) patterns
hourly_avg = df.groupby(df.index.hour)[target_col].mean()

plt.figure(figsize=(10, 4))
hourly_avg.plot(kind='bar', color='steelblue')
plt.title(f'Average {target_col} by Hour of Day')
plt.xlabel('Hour')
plt.ylabel(f'Average {target_col}')
plt.tight_layout()
plt.savefig('results/task1_hourly_pattern.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix — Air Quality Variables')
plt.tight_layout()
plt.savefig('results/task1_correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

**Observations**: Pollutant concentrations show clear daily cycles (typically higher during morning/evening rush-hour periods), and several gas sensor readings correlate strongly with each other (as expected, since related pollutants — e.g. CO and NOx — often originate from the same source, such as vehicle traffic).

## Part C — Feature Engineering

Created features and why they help:
- **Lag features** (t-1, t-2, t-3 hours): air quality has strong short-term autocorrelation — the most recent readings are usually the best predictor of the next reading.
- **Rolling mean** (3-hour, 24-hour): smooths out short-term sensor noise and captures the broader trend (e.g. a polluted day vs a clean day).
- **Hour of day**: captures repeating daily patterns like traffic rush hours.
- **Day of week**: captures weekday vs weekend differences in traffic/activity.
- **Month**: captures seasonal effects (e.g. heating in winter increasing certain pollutants).

In [ ]:
feature_df = df.copy()

# Lag features (using .shift() so only PAST values are used — avoids leakage)
for lag in [1, 2, 3]:
    feature_df[f'{target_col}_lag{lag}'] = feature_df[target_col].shift(lag)

# Rolling averages (also shifted by 1 so the current row's target isn't included)
feature_df[f'{target_col}_roll3'] = feature_df[target_col].shift(1).rolling(window=3).mean()
feature_df[f'{target_col}_roll24'] = feature_df[target_col].shift(1).rolling(window=24).mean()

# Time-based features
feature_df['hour'] = feature_df.index.hour
feature_df['day_of_week'] = feature_df.index.dayofweek
feature_df['month'] = feature_df.index.month

# Prediction target: next hour's value (t+1)
feature_df['target'] = feature_df[target_col].shift(-1)

# Drop rows with NaNs created by shifting/rolling
feature_df = feature_df.dropna()

print('Feature-engineered dataset shape:', feature_df.shape)
feature_df.head()

## Part D — Prediction

In [ ]:
feature_cols = [c for c in feature_df.columns if c != 'target']
X = feature_df[feature_cols]
y = feature_df['target']

# Time-respecting split: first 80% chronologically for training, last 20% for testing.
# We do NOT shuffle, since shuffling would let the model "see" future data during training.
split_idx = int(len(feature_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print('Train size:', X_train.shape, ' Test size:', X_test.shape)

In [ ]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    print(f'--- {name} ---')
    print(f'MAE:  {mae:.4f}')
    print(f'MSE:  {mse:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'R2:   {r2:.4f}\n')
    return model, preds, {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

lr_model, lr_preds, lr_metrics = evaluate_model('Linear Regression', LinearRegression(), X_train, y_train, X_test, y_test)
rf_model, rf_preds, rf_metrics = evaluate_model('Random Forest', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), X_train, y_train, X_test, y_test)

## Part E — Analysis

In [ ]:
# Actual vs Predicted plot (Random Forest, since it generally performs better)
plt.figure(figsize=(14, 5))
plt.plot(y_test.index, y_test.values, label='Actual', alpha=0.8)
plt.plot(y_test.index, rf_preds, label='Predicted (Random Forest)', alpha=0.8)
plt.title(f'Actual vs Predicted {target_col} (Test Period)')
plt.xlabel('Datetime')
plt.ylabel(target_col)
plt.legend()
plt.tight_layout()
plt.savefig('results/task1_actual_vs_predicted.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance from Random Forest
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(10).plot(kind='barh', color='seagreen')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('results/task1_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print(importances.head(10))

In [ ]:
# Where the model makes the largest errors
errors = np.abs(y_test.values - rf_preds)
worst_idx = np.argsort(errors)[-10:]

print('10 largest prediction errors occur at:')
for i in worst_idx:
    print(f'{y_test.index[i]}  Actual={y_test.values[i]:.2f}  Predicted={rf_preds[i]:.2f}  Error={errors[i]:.2f}')

**Data leakage discussion**: A key risk in time-series prediction is accidentally letting the model "see the future" during training. Two places this could happen here:
1. **Lag/rolling features**: if computed without `.shift()`, a rolling average at time *t* would include the value at *t* itself, which is essentially the value we're trying to predict — an obvious leak. We used `.shift(1)` before applying `.rolling()` so only strictly past values are used.
2. **Train/test split**: using a random shuffle split (like the default `train_test_split`) would let the model train on data that comes chronologically *after* some test points, letting the model implicitly learn from the future. We used a chronological (non-shuffled) 80/20 split to respect time order.

**Model performance**: Random Forest outperforms plain Linear Regression here (lower RMSE, higher R²), because it can capture non-linear relationships between the engineered features (lags, rolling averages, time-of-day) and the target, whereas Linear Regression assumes a purely linear relationship.

## Part F — Insights

### Key Findings
1. Pollutant levels show a strong daily cycle, with visible peaks around typical traffic rush hours.
2. The most recent lag features (t-1, t-2) are the strongest predictors of the next hour's reading, confirming strong short-term autocorrelation in air quality data.
3. Rolling averages (3-hour, 24-hour) help smooth sensor noise and improve prediction stability.
4. Random Forest clearly outperforms Linear Regression, suggesting non-linear relationships between features and pollutant levels.
5. Several pollutant readings (e.g. CO and NOx) are strongly correlated, likely due to a shared source (vehicle emissions).
6. The model's largest errors tend to occur during sudden spikes — sharp, short-lived pollution events that lag/rolling features can't fully anticipate.
7. Time-of-day and day-of-week features contribute meaningfully to prediction accuracy, confirming the value of cyclical time features.

### Limitations
1. The model doesn't have access to external factors like weather forecasts, traffic volume, or special events that can cause sudden pollution spikes.
2. A significant portion of the original dataset had missing values, and interpolation, while reasonable, can smooth over real short-term fluctuations.
3. The model predicts only 1 hour ahead — accuracy would likely degrade sharply for longer-horizon forecasts (e.g. 24 hours ahead).

### Possible Improvements
- Incorporate actual weather forecast data (not just historical temperature/humidity) and explore sequence models (e.g. LSTM) that can capture longer-range temporal dependencies better than lag-feature-based tree models.

In [ ]:
# Save final cleaned dataset and metrics
df.to_csv('results/task1_cleaned_air_quality.csv')

metrics = {
    'target_variable': target_col,
    'linear_regression': lr_metrics,
    'random_forest': rf_metrics
}

with open('results/task1_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Cleaned dataset and metrics saved.')
print(json.dumps(metrics, indent=2))